In [35]:
import requests
import json
import os
from openai import OpenAI


In [36]:

def get_reasoning_and_content(messages, 
                             model_type):
    """
    Sends chat messages to a vLLM server and returns a tuple of (reasoning_content, real_content).
    """
    if model_type == "ft":
        model="sukai/self_model"
        openai_api_key="EMPTY"
        openai_api_base="http://localhost:8001/v1"
    elif model_type == "raw":
        model="Qwen/Qwen2.5-7B-Instruct"
        openai_api_key="EMPTY"
        openai_api_base="http://localhost:8002/v1"
    else:
        model="Qwen/Qwen3-32B"
        openai_api_key="EMPTY"
        openai_api_base="http://localhost:8000/v1"
    
    client = OpenAI(
        api_key=openai_api_key,
        base_url=openai_api_base,
    )
    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    reasoning_content = ""
    real_content = ""
    start_reasoning_flg = False
    reasoning_inidicator = False
    count = 0

    for chunk in stream:
        count += 1
        # Handle reasoning content if present
        if chunk.choices and chunk.choices[0].delta and hasattr(chunk.choices[0].delta, "reasoning_content"):
            if not start_reasoning_flg and not reasoning_inidicator:
                print("=== Reasoning: ===")
                reasoning_inidicator = True
            reasoning_content += chunk.choices[0].delta.reasoning_content
            print(chunk.choices[0].delta.reasoning_content, end="", flush=True)
            start_reasoning_flg = True
        else:
            if count > 3 and start_reasoning_flg:
                print("=== End Reasoning ===\n")
                start_reasoning_flg = False
        # Handle main content
        if chunk.choices and chunk.choices[0].delta and chunk.choices[0].delta.content:
            real_content += chunk.choices[0].delta.content
            print(chunk.choices[0].delta.content, end="", flush=True)
    print()
    return reasoning_content, real_content


In [37]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "请帮我根据提示词，将相对应的关联词，关联词已经根据关联频率从高到低排好序了，请你将关联词分成四份，并进行造句"},
]

messages = [
    # {"role": "system", "content": "您是一款专为全面探索词语关联而设计的高级语言模型。"},
    {"role": "system", "content": "You are a helpful assistant."},
    
    {"role": "user", "content": "给定一个提示词，你的任务是生成一个与该提示词相关联的全面词汇列表。目标是尽可能涵盖所有相关的语境、用法和含义，避免重复相似的概念。不要生成受其他词语存在影响的词语，而是专注于提示词本身。提示词：方便。"},
]

print(" ===== test FT llm ====== ")

reasoning, content = get_reasoning_and_content(messages, 'ft')

print(" ===== test raw ====== ")
reasoning, content = get_reasoning_and_content(messages, 'raw')

print(" ===== test powerful =====")
reasoning, content = get_reasoning_and_content(messages, 'power')



 ===== test FT llm ====== 
， 单间， 快捷， 自助餐， 快， 快递， 高兴， 公共卫生间， 交通， 事项， 快速， 地铁， 有钱， 没人， 支付， 必需品， 自助， 插座， 自助超市， 教育， 道路， 休息， 回家， 避难所， 饭盒， 修改， 车， 厕所， 书， 甜饼干， 网银， 购物， 靠谱， 引申， 公厕， 通便， 日本， 吃的， 方圆， 开题报告， 交谈， 变通， 易上手， 睡觉， 轻松， 剪头， 优待， 写作， 通便剂， 意义， 好客， 简便， 度过， 买东西， 自助服务， 安全， 高效， 便宜， 避风， 简易， 上卫生间， 轻巧， 轻巧便利， 塑料袋， 顺手， 舒服， 简易， 索便， 卫生间， 巧， 问题， 市场， 帮助， 老弱病残， 要便， 开关， 方便面， 罗列， 不便， 顺利， 本身， 方便之门， 重要， 健身， 改善， 艺术， 实验， 面包， 常在
 ===== test raw ====== 
1. 便利
2. 轻松
3. 简便
4. 方便性
5. 便利性
6. 便捷
7. 不费劲
8. 自然
9. 天然
10. 轻松自如
11. 顺手
12. 轻松应对
13. 简化的问题
14. 简单易行
15. 不费吹灰之力
16. 易于使用
17. 易于获得
18. 好记忆
19. 良好的服务
20. 短暂快捷
21. 简化便捷
22. 时间管理
23. 生活高效
24. 性能好
25. 用起来方便
26. 简单好用
27. 出色的效率
28. 好用的工具
29. 打点上方便
30. 打点快速
31. 无摩擦
32. 简化复杂
33. 轻松控制
34. 易于调整
35. 轻便
36. 可携带
37. 好用的替代品
38. 好的解决方案
39. 轻松操作
40. 好用的特性
 ===== test powerful =====
=== Reasoning: ===

嗯，用户让我根据给定的提示词“方便”生成一个全面的词汇列表。首先，我需要理解用户的需求。他们想要一个涵盖所有相关语境、用法和含义的词汇库，避免重复或相似的概念。还要专注于“方便”本身，而不是被其他词影响。

首先，我得确定“方便”的中文含义。通常指方便快捷，容易处理或使用，但可能还有其他用法，比如方便面、方便袋这样的具体物品或服务。然后需要考虑不同领域中的用词，比如科

In [38]:
messages = [
    # {"role": "system", "content": "您是一款专为全面探索词语关联而设计的高级语言模型。"},
    {"role": "system", "content": "You are a helpful assistant."},
    
    {"role": "user", "content": "今天天气不错，去哪里玩好？"},
]

print(" ===== test FT llm ====== ")

reasoning, content = get_reasoning_and_content(messages, 'ft')

print(" ===== test raw ====== ")
reasoning, content = get_reasoning_and_content(messages, 'raw')

print(" ===== test powerful =====")
reasoning, content = get_reasoning_and_content(messages, 'power')



 ===== test FT llm ====== 
去公园散步、爬山或者去电影院看电影，也可以去逛逛街、美食街，品尝当地特色美食。
 ===== test raw ====== 
去哪里玩取决于您的兴趣和所在的城市或地区。如果您喜欢自然风光，可以考虑去公园或附近的山区散步；如果您喜欢文化氛围，可以考虑拜访当地的博物馆、展览馆或者历史遗址；如果您喜欢水上运动，可以去海边或湖边。北京的话可以去颐和园、圆明园，上海的话可以去闵行体育公园、共青森林公园等，您所在的城市是哪里呢？可以告诉我更多的信息，我可以为您推荐附近的游玩地点。
 ===== test powerful =====
=== Reasoning: ===

好的，用户问今天天气不错，去哪里玩好。我需要给出合适的建议。首先，天气好适合户外活动，但具体还要看用户的位置和兴趣。用户没有提供具体位置，可能需要假设一个常见的城市，比如北京、上海、成都这些热门城市，或者询问用户所在的位置。

接下来，考虑用户可能的兴趣点。常见的去处包括公园、动物园、博物馆、商场、逛街、郊外徒步、野餐、爬山、看电影等。可以分几个类别，比如自然景观、文化休闲、购物、美食探索，这样分类会更清晰。

同时也要注意天气适合的活动，比如晴天适合郊游、户外运动，但如果是毛毛雨或者有风，可能建议室内的地方。不过用户只说天气不错，可能没有其他细节，所以以晴天假设。

另外，是否需要推荐具体景点？比如北京的颐和园、上海的外滩、成都的锦里街等等。但可能用户并没有具体城市信息，所以需要保持通用建议，或者询问他们所在的位置。

还要考虑是否需要给出多个选择，让用户有选择的余地，或者是一种推荐和建议之间的平衡。例如，自然爱好者可能喜欢郊外，喜欢热闹的可能适合商场或美食街等。

可能的回答结构：1.确认天气是否适合户外；2.根据兴趣给出建议（室外活动和室内选择）；3.提供具体示例；4.询问他们所在城市，以提供更精准的推荐。

现在评估之前提供的回答是否符合这个思路：直接分几个方面，给出了一些示例景点，并提到如果需要具体推荐可以告诉位置。是的，这符合预期。但是是否可以更亲切、简短又富含合理建议呢，比如考虑不同时间段的活动建议？

最后，还需要用中文回答，并保持口语化和自然的风格，避免机械或生硬。
=== End Reasoning ===



今天天气不错的话，可